In [1]:
import kagglehub, os, glob, hashlib
IMG  = kagglehub.dataset_download("ashwanisingh997/ham10000-images-1229692")
SPL  = kagglehub.dataset_download("ashwanisingh997/ham10000-splits-1229692")
CODE = kagglehub.dataset_download("ashwanisingh997/thesis-code-1229692")
CKPT = kagglehub.dataset_download("ashwanisingh997/run1-checkpoints-1229692")
SYN  = kagglehub.dataset_download("ashwanisingh997/run1-synthetic-headline-1229692")

h = hashlib.sha256(open(f"{CODE}/metrics.py", "rb").read()).hexdigest().upper()
assert h == "602747135A78568EAF96E64DB70A7C43CAC6F8527089DBE26711B0D834F05E17", "metrics.py not D23 version"
src = {f: open(f"{CODE}/{f}").read() for f in ["checkpointing.py", "diffusion.py"]}
assert 'format={"jpg"' in src["checkpointing.py"] and "pipe.vae.enable_slicing()" in src["diffusion.py"]
print("code: v5 confirmed")

rows = []
for dp, _, files in os.walk(CKPT):
    for f in files:
        full = os.path.join(dp, f); hh = hashlib.sha256()
        with open(full, "rb") as fh:
            for ch in iter(lambda: fh.read(1 << 20), b""): hh.update(ch)
        rows.append((os.path.relpath(full, CKPT).replace(os.sep, "/"), hh.hexdigest()))
rows.sort()
dig = hashlib.sha256("\n".join(f"{p}|{s}" for p, s in rows).encode()).hexdigest()
assert dig == "81280427151b71c7eb1cc79d78415265d6924ec30c27210ee0bd0c708672dbf2", "CHECKPOINTS DIFFER"
print("checkpoints: MATCH")

SYN_ROOT = f"{SYN}/synthetic/run1_cloud"
ARMS = {"gan": "gan_final", "lora": "lora_final"}
for a in ARMS.values():
    assert len(glob.glob(f"{SYN_ROOT}/{a}/df/*.png")) == 1000
print("stored DF images 0-999: confirmed for both arms")

code: v5 confirmed
checkpoints: MATCH
stored DF images 0-999: confirmed for both arms


In [2]:
!pip uninstall -y -q torchao
import importlib; importlib.invalidate_caches()
import peft.import_utils as piu
if hasattr(piu.is_torchao_available, "cache_clear"): piu.is_torchao_available.cache_clear()
assert piu.is_torchao_available() is False
print("CELL 2 OK")

CELL 2 OK


In [3]:
import sys, shutil, gc, json, torch, yaml, numpy as np, pandas as pd
from PIL import Image
WORK = "/kaggle/working/repo"
os.makedirs(f"{WORK}/src", exist_ok=True); os.makedirs(f"{WORK}/config", exist_ok=True)
for f in os.listdir(CODE):
    shutil.copy(f"{CODE}/{f}", f"{WORK}/config/{f}" if f.endswith(".yaml") else f"{WORK}/src/{f}")
sys.path.insert(0, WORK); os.chdir(WORK)
print(torch.__version__, "| GPU:", torch.cuda.get_device_name(0))
assert torch.__version__.startswith("2.11")
cfg = yaml.safe_load(open("config/config.yaml"))
from src.run_registry import init_run
root = init_run("run1_cloud", root="/kaggle/working/results", git_hash="6c8939b")
from src.data import HAM10000, make_loader, build_image_index, assert_no_synthetic_leakage
assert_no_synthetic_leakage(f"{SPL}/val.csv", f"{SPL}/test.csv")
index = build_image_index(IMG)
NW = 4
val_loader  = make_loader(HAM10000(f"{SPL}/val.csv",  IMG, size=128, index=index), batch_size=32, shuffle=False, num_workers=NW)
test_loader = make_loader(HAM10000(f"{SPL}/test.csv", IMG, size=128, index=index), batch_size=32, shuffle=False, num_workers=NW)
print("setup OK")

2.11.0+cu128 | GPU: Tesla T4
[registry] run1_cloud (PRIMARY - Kaggle free-tier GPU, matches proposal Table 8.2)
[registry] segment 1 on Tesla T4 | torch 2.11.0+cu128 | cuda 12.8
[registry] writing to /kaggle/working/results/run1_cloud
[data] no synthetic leakage in 2 partition(s)
[data] indexed 10015 images under /kaggle/input/datasets/ashwanisingh997/ham10000-images-1229692
setup OK


In [4]:
from src.gan import Generator
from src.diffusion import BASE, build_pipeline, generate_diffusion, generate_gan_images
from diffusers import UNet2DConditionModel
from peft import PeftModel

SMOKE = False   # True: 1002 DF images and 2 one-epoch runs. False: the real run.
TARGET = 1002 if SMOKE else 2000
TAG = "_smoke" if SMOKE else ""
DFW = f"/kaggle/working/ablation_df{TAG}"
CK, LOG = f"{CKPT}/checkpoints", f"{root}/logs"

for a in ARMS.values():
    if not os.path.exists(f"{DFW}/{a}/df"):
        shutil.copytree(f"{SYN_ROOT}/{a}/df", f"{DFW}/{a}/df")

def repro(a):
    return [round(float(np.abs(np.asarray(Image.open(f"/kaggle/working/_repro/{a}/df/df_{i:04d}.png"), float)
            - np.asarray(Image.open(f"{SYN_ROOT}/{a}/df/df_{i:04d}.png"), float)).mean()), 3) for i in range(5)]

G = Generator(latent_dim=cfg["gan"]["latent_dim"])
G.load_state_dict(torch.load(f"{CK}/gan_final.pt", map_location="cpu", weights_only=False)["models"]["G"])
generate_gan_images(G, ["df"], 5, "/kaggle/working/_repro/gan_final", latent_dim=cfg["gan"]["latent_dim"], base_seed=42)
r = repro("gan_final"); print("GAN reproducibility:", r); assert max(r) < 2.0
generate_gan_images(G, ["df"], TARGET, f"{DFW}/gan_final", latent_dim=cfg["gan"]["latent_dim"],
                    base_seed=42, manifest_path=f"{LOG}/sampling_ablation_df_gan{TAG}.json")
del G; gc.collect(); torch.cuda.empty_cache()

unet = UNet2DConditionModel.from_pretrained(BASE, subfolder="unet", torch_dtype=torch.float16)
pipe = build_pipeline(PeftModel.from_pretrained(unet, f"{CK}/lora_final").merge_and_unload().to(torch.float16))
s = cfg["diffusion"]["sampler"]
kw = dict(base_seed=42, steps=s["num_inference_steps"], guidance=s["guidance_scale"],
          size=cfg["data"]["image_size_diffusion"], target_size=cfg["data"]["image_size_classifier"])
generate_diffusion(pipe, ["df"], 5, "/kaggle/working/_repro/lora_final", **kw)
r = repro("lora_final"); print("Diffusion reproducibility:", r); assert max(r) < 2.0
generate_diffusion(pipe, ["df"], TARGET, f"{DFW}/lora_final",
                   manifest_path=f"{LOG}/sampling_ablation_df_lora{TAG}.json", session_limit_hours=10.0, **kw)
del pipe, unet; gc.collect(); torch.cuda.empty_cache()
for a in ARMS.values():
    assert len(glob.glob(f"{DFW}/{a}/df/*.png")) == TARGET
print(f"DF images 0-{TARGET-1} ready for both arms")

[sample] {
  "out_dir": "/kaggle/working/_repro/gan_final",
  "classes": [
    "df"
  ],
  "n_per_class_target": 5,
  "counts_on_disk": {
    "df": 5
  },
  "base_seed": 42,
  "images_generated_this_session": 5,
  "images_skipped_existing": 0,
  "session_seconds": 0.74,
  "seconds_per_image": 0.147,
  "complete": true,
  "device": "Tesla T4"
}
GAN reproducibility: [0.0, 0.0, 0.0, 0.0, 0.0]
[sample] df: 50 made, 1000 skipped, 0.01s/image, ~0.00h left
[sample] df: 100 made, 1000 skipped, 0.01s/image, ~0.00h left
[sample] df: 150 made, 1000 skipped, 0.01s/image, ~0.00h left
[sample] df: 200 made, 1000 skipped, 0.01s/image, ~0.00h left
[sample] df: 250 made, 1000 skipped, 0.01s/image, ~0.00h left
[sample] df: 300 made, 1000 skipped, 0.01s/image, ~0.00h left
[sample] df: 350 made, 1000 skipped, 0.01s/image, ~0.00h left
[sample] df: 400 made, 1000 skipped, 0.01s/image, ~0.00h left
[sample] df: 450 made, 1000 skipped, 0.01s/image, ~0.00h left
[sample] df: 500 made, 1000 skipped, 0.01s/image, 

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

unet/diffusion_pytorch_model.safetensors: reconstructing file:   0%|          |  0.00B / 3.44GB            

unet/diffusion_pytorch_model.safetensors: downloading bytes:           |  0.00B            

There are modules in UNet2DConditionModel that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


... (3 outputs left out)

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


[sample] {
  "out_dir": "/kaggle/working/_repro/lora_final",
  "classes": [
    "df"
  ],
  "n_per_class_target": 5,
  "counts_on_disk": {
    "df": 5
  },
  "base_seed": 42,
  "images_generated_this_session": 5,
  "images_skipped_existing": 0,
  "session_seconds": 22.17,
  "seconds_per_image": 4.435,
  "complete": true,
  "device": "Tesla T4"
}
Diffusion reproducibility: [0.0, 0.0, 0.0, 0.0, 0.0]
[sample] df: 50 made, 1000 skipped, 5.34s/image, ~1.41h left
[sample] df: 100 made, 1000 skipped, 5.49s/image, ~1.37h left
[sample] df: 150 made, 1000 skipped, 5.55s/image, ~1.31h left
[sample] df: 200 made, 1000 skipped, 5.57s/image, ~1.24h left
[sample] df: 250 made, 1000 skipped, 5.59s/image, ~1.16h left
[sample] df: 300 made, 1000 skipped, 5.60s/image, ~1.09h left
[sample] df: 350 made, 1000 skipped, 5.61s/image, ~1.01h left
[sample] df: 400 made, 1000 skipped, 5.62s/image, ~0.94h left
[sample] df: 450 made, 1000 skipped, 5.62s/image, ~0.86h left
[sample] df: 500 made, 1000 skipped, 5.63s

In [5]:
from src.utils import set_seed
from src.classifier import build_model, train_classifier
from src.metrics import evaluate, save_metrics

SEEDS  = [42] if SMOKE else [42, 43, 44]
QTY    = [100] if SMOKE else [100, 250, 500, 1000, 2000]
EPOCHS = 1 if SMOKE else 30
PRE    = "SMOKE_" if SMOKE else ""

def df_rows(a, n):
    files = sorted(glob.glob(f"{DFW}/{a}/df/*.png"))[:n]
    assert len(files) == n
    return pd.DataFrame([{"image_id": os.path.basename(f)[:-4], "dx": "df", "path": f, "synthetic": True} for f in files])

for seed in SEEDS:
    for n in QTY:
        for arm, a in ARMS.items():
            name = f"{PRE}ABL_{arm}_df{n}_seed{seed}"
            if os.path.exists(f"{root}/metrics/{name}.json"):
                print(f"skip {name}"); continue
            print(f"\n{'='*60}\n{name}\n{'='*60}")
            set_seed(seed)
            tr = HAM10000(f"{SPL}/train.csv", IMG, size=128, augment=False, index=index, extra_rows=df_rows(a, n))
            assert len(tr) == 6981 + n, len(tr)
            m, _ = train_classifier(build_model(), make_loader(tr, batch_size=32, shuffle=True, num_workers=NW),
                                    val_loader, epochs=EPOCHS, lr=1e-4, weight_decay=1e-4, patience=7,
                                    tag=name, log_dir=f"{root}/logs")
            save_metrics(evaluate(m, test_loader, bootstrap=True, seed=seed), name, root=str(root))
            del m; torch.cuda.empty_cache()
print("ALL ABLATION RUNS COMPLETE")


ABL_gan_df100_seed42
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 183MB/s]


ep 00  loss 0.7898  val macroF1 0.5144  val acc 0.7866
ep 01  loss 0.3629  val macroF1 0.5406  val acc 0.7924
ep 02  loss 0.1617  val macroF1 0.5824  val acc 0.7963
ep 03  loss 0.0894  val macroF1 0.5965  val acc 0.7937
ep 04  loss 0.0587  val macroF1 0.6137  val acc 0.7931
ep 05  loss 0.0520  val macroF1 0.6029  val acc 0.7911
ep 06  loss 0.0409  val macroF1 0.6083  val acc 0.8094
ep 07  loss 0.0417  val macroF1 0.6074  val acc 0.7990
ep 08  loss 0.0242  val macroF1 0.6107  val acc 0.7931
ep 09  loss 0.0258  val macroF1 0.6195  val acc 0.7911
ep 10  loss 0.0208  val macroF1 0.6169  val acc 0.7977
ep 11  loss 0.0130  val macroF1 0.6308  val acc 0.8081
ep 12  loss 0.0150  val macroF1 0.6030  val acc 0.7990
ep 13  loss 0.0099  val macroF1 0.6018  val acc 0.7957
ep 14  loss 0.0077  val macroF1 0.6093  val acc 0.8009
ep 15  loss 0.0113  val macroF1 0.5988  val acc 0.8042
ep 16  loss 0.0050  val macroF1 0.5888  val acc 0.7866
ep 17  loss 0.0069  val macroF1 0.6100  val acc 0.7937
ep 18  los

In [6]:
rows = []
for n in QTY:
    for arm in ARMS:
        rs = [json.load(open(f"{root}/metrics/{PRE}ABL_{arm}_df{n}_seed{s}.json")) for s in SEEDS]
        for k, f in [("df_recall", lambda r: r["per_class"]["df"]["recall"]),
                     ("df_precision", lambda r: r["per_class"]["df"]["precision"]),
                     ("macro_f1", lambda r: r["macro_f1"]), ("melanoma_recall", lambda r: r["melanoma_recall"])]:
            v = [f(r) for r in rs]
            rows.append({"n_df": n, "arm": arm, "metric": k, "mean": np.mean(v),
                         "sd": np.std(v, ddof=1) if len(v) > 1 else float("nan"), "values": [round(x, 4) for x in v]})
summ = pd.DataFrame(rows).round(4)
summ.to_csv(f"{root}/metrics/{PRE}ablation_summary.csv", index=False)
print(summ.pivot_table(index="n_df", columns=["metric", "arm"], values="mean").to_string())
assert len(glob.glob(f"{root}/metrics/{PRE}ABL_*_predictions.npz")) == len(SEEDS) * len(QTY) * 2
shutil.make_archive("/kaggle/working/run1_ablation_results", "zip", "/kaggle/working/results")
print("saved: /kaggle/working/run1_ablation_results.zip")

metric df_precision         df_recall         macro_f1         melanoma_recall        
arm             gan    lora       gan    lora      gan    lora             gan    lora
n_df                                                                                  
100          0.5500  0.5680    0.5833  0.5167   0.6679  0.6542          0.4571  0.4611
250          0.4374  0.4980    0.4833  0.5833   0.6181  0.6324          0.4152  0.4471
500          0.5079  0.4860    0.4833  0.6500   0.6266  0.6490          0.5489  0.4731
1000         0.4699  0.5292    0.4833  0.5333   0.6344  0.6651          0.4850  0.5010
2000         0.5536  0.6553    0.3000  0.5333   0.6232  0.6684          0.5449  0.4731
saved: /kaggle/working/run1_ablation_results.zip
